# EXP006 CUDA migration — Colab runner
Use this notebook only after applying the versioned EXP006 device amendment locally and creating `EXP006_colab_workspace.zip`. The transfer bundle contains TRAIN arrays only; NASA TEST arrays and anomaly labels are intentionally excluded.


## 0. Enable a GPU
In Colab choose **Runtime → Change runtime type → GPU** before running anything below.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import zipfile, shutil, os

ZIP_PATH = Path('/content/drive/MyDrive/EXP006_colab_workspace.zip')  # edit if needed
WORKSPACE = Path('/content/drive/MyDrive/EXP006_colab_workspace')
PROJECT = WORKSPACE / 'workspace' / 'smap_msl_tcn_gcn_v2'

assert ZIP_PATH.exists(), f'Missing {ZIP_PATH}'
if not PROJECT.exists():
    WORKSPACE.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
        zf.extractall(WORKSPACE)
print('PROJECT =', PROJECT)
print('Exists:', PROJECT.exists())


In [ ]:
import torch, numpy as np, pandas as pd, sys
print('Python:', sys.version)
print('PyTorch:', torch.__version__)
print('NumPy:', np.__version__)
print('pandas:', pd.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('CUDA:', torch.version.cuda)
    print('cuDNN:', torch.backends.cudnn.version())
assert torch.cuda.is_available(), 'GPU runtime is not enabled.'


## 1. Amendment-aware preflight
This checks the original EXP006 freeze, the versioned device amendment, the 81-channel TRAIN study set, candidate definitions, and the requested CUDA backend.


In [ ]:
os.chdir(PROJECT)
!python src/v2_run_exp006.py --stage preflight --device cuda --workers 1


## 2. CPU↔CUDA bridge verification
This runs the bounded verification approved before migration: exact identity checks, CPU regression, numerical parity, four CPU/CUDA full-training bridge pairs, two GPU repeats, and checkpoint portability. It writes `exp006_device_verification.csv` and `exp006_execution_environment.json`.


In [ ]:
!python src/v2_verify_exp006_device.py


In [ ]:
import pandas as pd
verification = pd.read_csv(PROJECT / 'exp006_device_verification.csv')
display(verification)
print('Failures:', int((~verification['passed'].astype(bool)).sum()))


### STOP HERE
Send the verification summary/output back for audit before launching robustness. Do not continue if any verification row failed.


## 3. EXP006 chronological robustness — run only after verification audit
All 192 fits, including refitted EXP002 and EXP005 controls, run under the same CUDA environment. The runner is resumable because each completed fit is written to Drive.


In [ ]:
!python src/v2_run_exp006.py --stage robust --device cuda --workers 1


### STOP AGAIN
Send the robustness summary back for audit before confirmation.


## 4. EXP006 full confirmation — run only after robustness audit
This deliberately recomputes all 486 confirmation fits on CUDA and does not substitute CPU screening checkpoints.


In [ ]:
!python src/v2_run_exp006.py --stage confirm --device cuda --workers 1


## 5. Promotion decision — only after confirmation audit


In [ ]:
!python src/v2_run_exp006.py --stage promote --device cuda --workers 1
